In [0]:
from pyspark.sql.functions import current_timestamp, col, lit

# Point to offline snapshot in the UC Volume
snapshot_path = "/Volumes/railway_analytics/bronze/raw_landing/live_train_snapshot_20260523.csv"

print("Running in OFFLINE mode. Fetching static snapshot...")

# Load the snapshot as strings to bypass inferSchema issues
live_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(snapshot_path)

# Cast the numeric column required by the Bronze DDL
if "delay_mins" in live_df.columns:
    live_df = live_df.withColumn("delay_mins", col("delay_mins").cast("int"))

# Add ingestion metadata
if "ingested_at" not in live_df.columns:
    live_df = live_df.withColumn("ingested_at", current_timestamp())

# Tag the source system
if "data_source" not in live_df.columns:
    live_df = live_df.withColumn("data_source", lit("NTES_Snapshot_Offline"))

# Append to the managed Bronze table
live_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.live_train_status")

print("Offline ingestion complete! Bronze layer updated.")

Running in OFFLINE mode. Fetching static snapshot...
Offline ingestion complete! Bronze layer updated.
